## Imports and utilitaries

Run all cells in this section before simulating.


In [6]:
# copied from all_possible_paths.py
 
### In this script I can try all the different combinations of actions for the AV agents.
import itertools
import os
import pandas as pd
import numpy as np
import csv
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torchrl.envs.libs.pettingzoo import PettingZooWrapper
from torchrl.envs.transforms import TransformedEnv, RewardSum
from torchrl.envs.utils import check_env_specs
from torch import nn
from torchrl._utils import logger as torchrl_logger
from torchrl.collectors import SyncDataCollector
from torchrl.data import TensorDictReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.modules import EGreedyModule, QValueModule, SafeSequential
from torchrl.modules.models.multiagent import MultiAgentMLP
from torchrl.objectives import SoftUpdate, ValueEstimators, DQNLoss

from routerl import TrafficEnvironment

from routerl.keychain import Keychain as kc
from routerl.utilities import get_params

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [ ]:
def update_trafficlight(t0,t1,ty):

    read_file_name = "../networks/two_route_trafficlight/two_route_trafficlight.net.xml"
    write_file_name = "../venv/Lib/site-packages/routerl/networks/two_route_trafficlight/two_route_trafficlight.net.xml"
    network_file_name = "../networks/two_route_trafficlight/two_route_trafficlight.net.xml"

    tllogic = []
    tllogic.append( "\t<tlLogic id=\"J2\" type=\"static\" programID=\"0\" offset=\"0\">\n")
    tllogic.append( "\t\t<phase duration=\"%s\" state=\"Gr\"/>\n"%(t1))
    tllogic.append( "\t\t<phase duration=\"%s\"  state=\"yr\"/>\n"%(ty))
    tllogic.append( "\t\t<phase duration=\"%s\" state=\"rG\"/>\n"%(t0))
    tllogic.append( "\t\t<phase duration=\"%s\"  state=\"ry\"/>\n"%(ty))
    tllogic.append( "\t</tlLogic>\n")

    with open(read_file_name, "r") as fr:
        lines = fr.readlines()
        print(lines)

    with open(write_file_name, "w") as fw:
        fw.writelines(lines[:76])
        fw.writelines(tllogic)
        fw.writelines(lines[82:])

    with open(network_file_name, "w") as fw:
        fw.writelines(lines[:76])
        fw.writelines(tllogic)
        fw.writelines(lines[82:])
    
    return None

def create_environment(nb_agents=23):
    env = TrafficEnvironment(
        agent_parameters={
            "num_agents": nb_agents, 
            "new_machines_after_mutation": 10, 
            "machine_parameters": {
                "behavior": "selfish"
                }
            },
        simulator_parameters={
            "sumo_type": "sumo"
            },
        path_generation_parameters={
            "number_of_paths": 2
            }
        )
    return env

def simulate(nb_agents=23):

    env = create_environment(nb_agents)
    env.start()
    env.mutation()

    actions = [0, 1]
    print("env.human_agents", env.human_agents)
    print("env.machine_agents", env.machine_agents)
    print("\n")

    env.reset()

    i = 0
    k = 1
    for combination in itertools.product(actions, repeat=len(env.possible_agents)):
        i += 1
        for action in combination:
            env.step(action)
        if i > k*1024/10:
            print("%s combinations out of 1024 tested, %s%s remaining"%(i,(10-k)*10,"%"))
            k += 1

        env.reset()
    
    env.stop_simulation()

def build_df(i):
    df = pd.read_csv("training_records/episodes/ep"+str(i)+".csv")
    df = df[df["kind"] == "AV"]
    df = df.sort_values(by="start_time").reset_index(drop=True)
    df["reward"] = -1*df["travel_time"]
    df = df[["reward","action"]]
    return df

def write_line(i,df):
    line = str(i)
    for i in range(10):
        line = line + "," + str(df["reward"].values[i])
    return line

def record_experiment(file_name):
    with open(file_name, "w") as f:
        f.write("id,0,1,2,3,4,5,6,7,8,9\n")
        for i in range(1024):
            data = build_df(i+1)
            text = write_line(i,data)
            f.write(text+"\n")

In [ ]:
def id_to_strategy(id):
    strategy = [0 for _ in range(10)]
    i = 0
    while id > 0:
        if id % 2 == 1:
            strategy[9-i] = 1
        id = id//2
        i += 1
    return strategy

def strategy_to_id(s):
    id = 0
    for i in range(10):
        id += s[i]*(2**(9-i))
    return id

## Simulation

`run(tl_0, tl_1, tl_y, nbagents)` :
* generates a two-route-trafficlight network with the specified traffic light times (writes a .net.xml file **both** within the virtual environment (for execution), and within the ..\networks\two_route_trafficlight folder (for safekeeping). Make sure both paths exist on the machine otherwise it won't run.);
* initializes the SUMO environment with the specified number of agents;
* runs the simulation for all 1024 possible AV joint actions;
* writes the payoff matrix inside the `reward_df_(tl_0)_(tl_1)_(tl_y)_(nbagents)agents.csv` file.

In [14]:
def run(tl_0, tl_1, tl_y, nbagents):
    filename = "reward_df_%s_%s_%s_%sagents.csv"%(tl_0,tl_1, tl_y, nbagents)
    update_trafficlight(tl_0, tl_1, tl_y)
    simulate(nbagents)
    record_experiment(filename)

In [15]:
run(20,30,5,23)

['<?xml version="1.0" encoding="UTF-8"?>\n', '\n', '<!-- generated on 2025-04-16 16:19:46 by Eclipse SUMO netedit Version 1.22.0\n', '<neteditConfiguration xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="http://sumo.dlr.de/xsd/neteditConfiguration.xsd">\n', '\n', '    <input>\n', '        <sumo-net-file value="C:\\Users\\Utilisateur\\Desktop\\projectrouterl\\venv\\Lib\\site-packages\\routerl\\networks\\two_route_trafficlight\\two_route_trafficlight.net.xml"/>\n', '    </input>\n', '\n', '    <output>\n', '        <output-file value="C:\\Users\\Utilisateur\\Desktop\\projectrouterl\\venv\\Lib\\site-packages\\routerl\\networks\\two_route_trafficlight\\two_route_trafficlight.net.xml"/>\n', '    </output>\n', '\n', '    <processing>\n', '        <geometry.min-radius.fix.railways value="false"/>\n', '        <geometry.max-grade.fix value="false"/>\n', '        <offset.disable-normalization value="true"/>\n', '        <lefthand value="0"/>\n', '    </proce

In [16]:
for i in [10,15,20,25,30]:
    tl_0, tl_1, tl_y = i, 50-i, 5
    nbagents = 20
    update_trafficlight(tl_0, tl_1, tl_y)
    simulate()
    filename = "reward_df_%s_%s.csv"%(tl_0,tl_1)
    record_experiment(filename)

['<?xml version="1.0" encoding="UTF-8"?>\n', '\n', '<!-- generated on 2025-04-16 16:19:46 by Eclipse SUMO netedit Version 1.22.0\n', '<neteditConfiguration xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="http://sumo.dlr.de/xsd/neteditConfiguration.xsd">\n', '\n', '    <input>\n', '        <sumo-net-file value="C:\\Users\\Utilisateur\\Desktop\\projectrouterl\\venv\\Lib\\site-packages\\routerl\\networks\\two_route_trafficlight\\two_route_trafficlight.net.xml"/>\n', '    </input>\n', '\n', '    <output>\n', '        <output-file value="C:\\Users\\Utilisateur\\Desktop\\projectrouterl\\venv\\Lib\\site-packages\\routerl\\networks\\two_route_trafficlight\\two_route_trafficlight.net.xml"/>\n', '    </output>\n', '\n', '    <processing>\n', '        <geometry.min-radius.fix.railways value="false"/>\n', '        <geometry.max-grade.fix value="false"/>\n', '        <offset.disable-normalization value="true"/>\n', '        <lefthand value="0"/>\n', '    </proce

In [17]:
##################### Environment Creation #####################
# params = get_params("params.json")
# print(params[kc.PLOTTER][kc.RECORDS_FOLDER])

# env.start()
###################### Human Learning #####################
#num_episodes = 0
#
#for episode in range(num_episodes):
#    env.step()
#
##################### Mutation #####################
#env.mutation()